# Retail Customer Churn Prediction with ANN (MLP)

## Objectives
- Predict whether a telecom retail customer will **churn** (leave).
- Build an end-to-end **feedforward neural network** (multi-layer perceptron).

## Theory (Simple English)
An **ANN** stacks layers of neurons. Each neuron computes a weighted sum of inputs plus bias, then applies an **activation** (ReLU for hidden layers, sigmoid for binary output).

**Forward pass:** $z = Wx + b$, $a = \text{ReLU}(z)$

**Loss (binary classification):** Binary cross-entropy compares predicted probability to true label 0/1.

**Why MLP here?** Tabular features (tenure, charges, contract type) map well to dense layers after encoding categoricals.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Problem Definition & Business Context

**Churn** costs retailers and telcos heavily (acquisition > retention).  
**Goal:** Flag high-risk customers early for retention offers.  
**Success metric:** Recall on churners (catch leavers) with acceptable precision.


In [ ]:
import io
import requests

# Telco Customer Churn — public mirror (IBM sample, stable CSV structure)
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
r = requests.get(URL, timeout=60)
r.raise_for_status()
df = pd.read_csv(io.StringIO(r.text))
print(df.shape)
df.head()


In [ ]:
# EDA
print(df.info())
print(df["Churn"].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Churn distribution")
if "MonthlyCharges" in df.columns:
    sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
    axes[1].set_title("Monthly Charges vs Churn")
plt.tight_layout()
plt.show()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Numeric feature correlations")
    plt.show()


In [ ]:
# Preprocessing
df = df.drop(columns=["customerID"], errors="ignore")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
y = df["Churn"].values
X_df = df.drop(columns=["Churn"])

# One-hot encode categoricals
X_df = pd.get_dummies(X_df, drop_first=True)
X = X_df.values.astype(np.float32)
feature_names = list(X_df.columns)
print("Features:", len(feature_names))


In [ ]:
# Train / validation / test split (stratified for classification)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)  # ~70/15/15

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Val:", X_val_s.shape, "Test:", X_test_s.shape)


In [ ]:
# Build MLP (ANN)
model = models.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)
model.summary()


In [ ]:
checkpoint = callbacks.ModelCheckpoint(
    "ann_churn_best.keras", monitor="val_loss", save_best_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
cb_list = [checkpoint, early_stop, reduce_lr]


In [ ]:
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=80,
    batch_size=32,
    callbacks=cb_list,
    verbose=1,
)

# Loss curves
pd.DataFrame(history.history).plot(figsize=(10, 4))
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.show()

# Test metrics
y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
try:
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
except Exception:
    pass
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Test)")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()


In [ ]:
# Inference demo on a few test rows
loaded = keras.models.load_model("ann_churn_best.keras")
sample_idx = np.arange(min(5, len(X_test_s)))
samples = X_test_s[sample_idx]
probs = loaded.predict(samples, verbose=0).ravel()
for i, p in zip(sample_idx, probs):
    print(f"Row {i} -> P(positive class): {p:.4f}, predicted: {int(p >= 0.5)}, actual: {int(y_test[i])}")

import joblib
joblib.dump(scaler, "scaler.pkl")
print("Saved scaler.pkl and ann_churn_best.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
